# SDG 3: Good health and well-being

The SDG 3 (Good health and well-being) index of the SDG Atlas of Bolivia 2026 and its 7 indicators, skilled birth attendance, infant and under-5 mortality, HIV, tuberculosis and vector-borne diseases, and adolescent fertility, for every municipality near 2012 and near 2024.

This notebook gathers every code box of the indicator's page, [https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities343-sdg03/](https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities343-sdg03/) (GeoDevelopment Observatory of Bolivia), so you can work with the data in one place. Run **Setup** first; then any section runs on its own (Runtime › Run all runs everything).

## Ask Gemini, Colab's AI

Colab has an AI assistant, Gemini. Open it with the Gemini button (✦) in Colab's toolbar, or click **Generate** in an empty code cell, and paste one of these questions. It writes and explains code that builds on the data loaded below. Run Setup first, and check its answers against the variables' descriptions on the page.

- "Map infant mortality (sdg3_2_tmi) in 2024 and list the 10 municipalities with the highest rates."
- "How much did under-5 mortality (sdg3_2_tmn) fall between the two periods? Show the change per municipality and the median by department."
- "Plot births attended by skilled health personnel (sdg3_1_cpp) against infant mortality (sdg3_2_tmi) in 2024 and describe the relationship."
- "Where is tuberculosis incidence (sdg3_3_tub) highest? Map it for 2024 and check whether it clusters spatially with geo.moran."
- "Infant mortality's actual years are 2008 and 2023, not 2012 and 2024. Use geo.reference_years("sdg03") to list every variable's years and explain how to describe the change correctly."


## Variables

The columns of `sdg03`, one row per unit and year. Each one's full description (how it was built, its source and reference years) is under Variables on the page, and in `geo.dictionary("sdg03")`.

| Variable | Label | Unit |
| --- | --- | --- |
| `geo_id` | Municipality ID |  |
| `year` | Period |  |
| `mun_id` | INE municipality code |  |
| `index_sdg3` | Good health and well-being index (0-100) | index (0-100) |
| `sdg3_1_cpp` | Births attended by skilled health personnel (%) | % of births |
| `sdg3_2_tmi` | Infant mortality (per 1,000 live births) | deaths under 1 per 1,000 live births |
| `sdg3_2_tmn` | Under-5 mortality (per 1,000 live births) | deaths under 5 per 1,000 live births |
| `sdg3_3_vih` | HIV incidence (per million people) | cases per million people |
| `sdg3_3_tub` | Tuberculosis incidence (per 100,000 people) | cases per 100,000 people |
| `sdg3_3_vec` | Vector-borne disease index (0-1) | index (0-1) |
| `sdg3_7_tfa` | Mean parity of adolescent women 15-19 (children per woman) | live births per woman 15-19 |


## Setup

Installs the packages every section uses and imports them once.

In [ ]:
!pip install -q geo4bolivia geolibre
import geo4bolivia as geo
import plotly.express as px
import pandas as pd
import geopandas as gpd
from geolibre import Map

## Load the data with the package

In [ ]:
# The SDG 3 index (0-100, higher is better), ~2024
gdf = geo.load("sdg03", variables="index_sdg3", years=2024, geometry=True)
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg3", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg3": "Good health and well-being index (0-100)"},
          title="SDG 3: Good health and well-being index, ~2024").show()

# The two periods: initial (~2012) and final (~2024) values
df = geo.load("sdg03", variables="index_sdg3")
wide = df.pivot(index="geo_id", columns="year", values="index_sdg3")
wide = wide.add_prefix("y")
wide["name"] = geo.units("map343").set_index("geo_id")["name"]
px.scatter(wide, x="y2012", y="y2024", hover_name="name",
           labels={"y2012": "Index, ~2012", "y2024": "Index, ~2024"},
           title="SDG 3: Good health and well-being index, ~2012 and ~2024")

## Load the data without the package (plain URLs)

In [ ]:
# No geo4bolivia needed: every file is a plain URL
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
df = pd.read_csv(BASE + "data/sdg03/map343/sdg03.csv")  # geo_id, year, mun_id, ...
municipalities = gpd.read_file(BASE + "geo/boundaries/map343.geojson")

index = df.loc[df["year"] == 2024, ["geo_id", "index_sdg3"]]
gdf = municipalities.merge(index, on="geo_id")
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg3", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg3": "Good health and well-being index (0-100)"},
          title="SDG 3: Good health and well-being index, ~2024")

## Load the data from the wide GeoPackage (one row per municipality)

In [ ]:
# No geo4bolivia needed: one row per municipality, a column per variable
# and year (<variable>_<year>), with boundaries
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
gdf = gpd.read_file(BASE + "data/sdg03/map343/sdg03_wide.gpkg",
                    layer="sdg03")   # or sdg03_wide.geojson (lighter)
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg3_2024", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg3_2024": "Good health and well-being index (0-100)"},
          title="Good health and well-being index (0-100), ~2024")

## Load the data in GeoLibre (the wide GeoJSON)

In [ ]:
# GeoLibre in the notebook: the wide GeoJSON, one feature per municipality
# and a column per variable and year (<variable>_<year>); restyle any of
# them in the Style panel
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
m = Map(center=[-64.5, -16.5], zoom=4.5, height="600px")
m.add_choropleth(BASE + "data/sdg03/map343/sdg03_wide.geojson",
                 column="index_sdg3_2024", name="Good health and well-being index (0-100), ~2024",
                 strokeColor="#ffffff", strokeWidth=0.3)
m

## Map the change, ~2012 to ~2024

In [ ]:
df = geo.load("sdg03", variables="index_sdg3")
wide = df.pivot(index="geo_id", columns="year", values="index_sdg3")
wide["change"] = wide[2024] - wide[2012]
gdf = geo.boundaries("map343").merge(wide["change"].reset_index(),
                                           on="geo_id")
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="change", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"change": "Change in the index (points)"},
          title="SDG 3: change in the index, ~2012 to ~2024")

## Recompute the index from its indicators

In [ ]:
df = geo.load("sdg03", years=2024).set_index("geo_id")
bounds = {"sdg3_1_cpp": (0, 100), "sdg3_2_tmi": (39.7, 1.1), "sdg3_2_tmn": (130.1,
          2.6), "sdg3_3_vih": (721.5, 0), "sdg3_3_tub": (199.2, 0),
          "sdg3_3_vec": (0.68, 0), "sdg3_7_tfa": (0.71, 0)}  # (worst, best)
scores = pd.DataFrame({v: 100 * (df[v] - w) / (b - w)
                       for v, (w, b) in bounds.items()}).clip(0, 100)
df["computed"] = scores.mean(axis=1)  # skips missing indicators
df["name"] = geo.units("map343").set_index("geo_id")["name"]
px.scatter(df, x="computed", y="index_sdg3", hover_name="name",
           labels={"computed": "Computed from the indicators",
                   "index_sdg3": "Good health and well-being index (0-100)"},
           title="Recomputed vs published SDG 3 index, ~2024")

## Which indicators moved

In [ ]:
df = geo.load("sdg03")
bounds = {"sdg3_1_cpp": (0, 100), "sdg3_2_tmi": (39.7, 1.1), "sdg3_2_tmn": (130.1,
          2.6), "sdg3_3_vih": (721.5, 0), "sdg3_3_tub": (199.2, 0),
          "sdg3_3_vec": (0.68, 0), "sdg3_7_tfa": (0.71, 0)}  # (worst, best)
labels = geo.dictionary("sdg03").set_index("variable")["label"]
rows = [{"indicator": labels[v], "period": f"~{year}",
         "score": (100 * (part[v] - w) / (b - w)).clip(0, 100).mean()}
        for year, part in df.groupby("year") for v, (w, b) in bounds.items()]
px.scatter(pd.DataFrame(rows), x="score", y="indicator", color="period",
           labels={"score": "Mean rescaled score (0-100)", "indicator": ""},
           title="SDG 3 indicators, ~2012 and ~2024")

## Rank the departments

In [ ]:
df = geo.load("sdg03", variables="index_sdg3")
dep = geo.aggregate(df, to="department",
                    rules={"index_sdg3": "weighted_mean:population"},
                    weights=geo.load("sdg", variables="population"))
dep["department"] = dep["geo_id"].map(
    geo.units("department").set_index("geo_id")["name"])
dep["period"] = "~" + dep["year"].astype(str)
px.scatter(dep.sort_values("index_sdg3"), x="index_sdg3", y="department",
           color="period",
           labels={"index_sdg3": "Index (population-weighted)"},
           title="SDG 3 index by department, ~2012 and ~2024")

## License and how to cite

The data are published under [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/); their sources keep their own licenses:

- Lykke E. Andersen and Fabiana Argandoña (SDSN Bolivia, Universidad Privada Boliviana): [Creative Commons Attribution 4.0 International License (licensed by SDSN Bolivia)](https://creativecommons.org/licenses/by/4.0/)
- SDSN Bolivia: [Creative Commons Attribution 4.0 International License (licensed by SDSN Bolivia)](https://creativecommons.org/licenses/by/4.0/)

If you use these data, please cite, in this order, the original source, the observatory, which processes and maintains them, and the other sources:

1. Andersen, L. E., & Argandoña, F. (2026). Atlas de los ODS en Bolivia 2026: Brechas y tendencias del desarrollo sostenible en las Entidades Territoriales Autónomas, 2012–2024. La Paz: Universidad Privada Boliviana, SDSN Bolivia.
2. Mendez, C. (2026). GeoDevelopment Observatory of Bolivia. https://quarcs-lab.github.io/gdo4bolivia/
3. SDSN Bolivia (2025). Límites Municipales Bolivia 2025 [Archivo shapefile]. https://sdsnbolivia.org/datos-espaciales/
